In [16]:
%%time
import json
import re
import warnings

import pandas as pd
import penman
import torch
from smatch import score_amr_pairs
from tqdm import tqdm
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration
from transformers.utils import logging

logging.set_verbosity_error()
warnings.filterwarnings("ignore")

CPU times: user 324 μs, sys: 3 μs, total: 327 μs
Wall time: 329 μs


In [2]:
%%time
model_name = "BramVanroy/mbart-large-cc25-ft-amr30-en"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

CPU times: user 2.65 s, sys: 2.29 s, total: 4.93 s
Wall time: 16 s


MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(253271, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(253271, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True

In [25]:
def linearized_to_penman(linearized, graph_idx=0):
    """
    Convert linearized AMR to Penman format safely.
    Ensures unique node names per graph to avoid SMATCH errors.

    Args:
        linearized: str, linearized AMR
        graph_idx: int, unique index for this graph (used to prefix variable names)
    Returns:
        str, Penman-formatted AMR
    """
    text = linearized.replace("<AMR>", "").strip()
    pointer_map = {}
    var_idx = 0

    def replace_pointer(match):
        nonlocal var_idx
        idx = match.group(1)
        if idx not in pointer_map:
            # Use a unique prefix per graph to avoid duplicates
            pointer_map[idx] = f"x{graph_idx}_{var_idx}"
            var_idx += 1
        return f"({pointer_map[idx]} / "

    # Replace all <pointer:NUMBER> with unique variable names
    text = re.sub(r"<pointer:(\d+)>", replace_pointer, text)

    # Replace relation tags with parentheses
    text = text.replace("<rel>", " ")
    text = text.replace("</rel>", ")")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Balance parentheses
    open_par = text.count("(")
    close_par = text.count(")")
    text += ")" * (open_par - close_par)

    return text

In [4]:
df = pd.read_csv("../baseline/data.csv")
df.shape

(1100, 2)

In [5]:
df.head()

,sentence,amr_penman
0,मुरगाव पालिकेच्या 239 लाखांच्या शिलकी अर्थसंकल...,(a / approve-01 :ARG1 (b / budget :poss (g / g...
1,डिझेलाच्या दरांत प्रति लिटर 80 पयशांनी वाड जाल्या,(i / increase-01\n :ARG1 (p / price\n ...
2,पुराय देशभरांत हे दर वाडयल्यात,(i / increase-01\n :ARG1 (r / rate\n :m...
3,राजकी परिस्थिती खिणाखिणाक बदलत आसा,(c / change-01\n :ARG1 (s / situation\n ...
4,वाडटे म्हारगाये आड काग्रेसीचें आंदोलन,(p / protest-01\n :ARG0 (c / political-party...


In [6]:
sentences = df["sentence"].tolist()
gold_amrs = df["amr_penman"].tolist()

In [9]:
%%time
results = []

for sent, gold in tqdm(zip(sentences, gold_amrs), total=len(sentences)):
    tokenizer.src_lang = "hi_IN"  # best for Konkani

    inputs = tokenizer(sent, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"],
        max_length=256,
        num_beams=4,
    )

    raw_output = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    pred_penman = linearized_to_penman(raw_output)

    results.append(
        {
            "sentence": sent,
            "gold_amr": gold.strip(),
            "model_output_linearized": raw_output,
            "model_output_penman": pred_penman.strip(),
        }
    )

100%|██████████| 1100/1100 [24:19<00:00,  1.33s/it]

CPU times: user 24min 7s, sys: 690 ms, total: 24min 8s
Wall time: 24min 19s


In [10]:
with open("konkani_amr_predictions.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

In [19]:
# # %%time
# # gold_graphs = []
# # pred_graphs = []

# # for r in results:
# #     try:
# #         gold_graphs.append(penman.decode(r["gold_amr"]))
# #         pred_graphs.append(penman.decode(r["model_output_penman"]))
# #     except:
# #         continue
# gold_graphs = []
# pred_graphs = []

# for r in results:
#     try:
#         g_gold = penman.decode(r["gold_amr"])
#         g_pred = penman.decode(r["model_output_penman"])
#         if g_gold is None or g_pred is None:
#             continue  # skip invalid graphs
#         gold_graphs.append(g_gold)
#         pred_graphs.append(g_pred)
#     except Exception as e:
#         # print("Skipping invalid graph:", e)
#         pass
#         # continue

In [27]:
gold_graphs = []
for idx, r in enumerate(results):
    try:
        g_gold = penman.decode(r["gold_amr"])
        if g_gold is not None:
            gold_graphs.append(g_gold)
        else:
            # print(f"Skipping invalid gold AMR at index {idx}")
            pass
    except Exception:
        # print(f"Skipping gold AMR at index {idx}: {e}")
        pass

pred_graphs = []
for idx, r in enumerate(results):
    try:
        pred_amr_text = linearized_to_penman(r["model_output_penman"], graph_idx=idx)
        g_pred = penman.decode(pred_amr_text)
        if g_pred is not None:
            pred_graphs.append(g_pred)
        else:
            # print(f"Skipping invalid predicted AMR at index {idx}")
            pass
    except Exception:
        # print(f"Skipping predicted AMR at index {idx}: {e}")
        pass

aligned_gold = []
aligned_pred = []

for g, p in zip(gold_graphs, pred_graphs):
    if g is not None and p is not None:
        aligned_gold.append(g)
        aligned_pred.append(p)

print(f"Using {len(aligned_gold)} valid AMR pairs for SMATCH")

ignoring epigraph data for duplicate triple: ('s', ':instance', 'some')
Missing concept: (x0 / say-01:ARG0 (x1 / i):ARG1 (x2 / अर्ज-01:mode imperative:ARG0(x3 / :ARG1(x4 / ):ARG2 (x4 / person:quant 18:wiki -:name (x3 / name:op1<lit> चो नगरसेवक</lit>:op2<lit> चोपडेंकाराचो</lit>:op3<lit> अर्ज</lit>)))))
Missing concept: (x0 / say-01:ARG0 (x1 / i):ARG1 (x2 / अर्ज-01:mode imperative:ARG0(x3 / :ARG1(x4 / ):ARG2 (x4 / person:quant 18:wiki -:name (x3 / name:op1<lit> चो नगरसेवक</lit>:op2<lit> चोपडेंकाराचो</lit>:op3<lit> अर्ज</lit>)))))
Missing target: (x0 / cause-01:ARG0 (x1 / continue-01:negation:ARG0 (x2 / you):time (x3 / date-entity:month 12:year 1855)):ARG1 (x4 / person:wiki -:name (x5 / name:op1<lit> मरसर</lit>:op2<lit> Pascal</lit>:op3<lit> Sitri</lit>):ARG0-of (x6 / have-org-role-91:ARG1 (x7 / government-organization:wiki<lit> United States Department of Agriculture</lit>:name (x8 / name:op1<lit> Agriculture</lit>))):ARG0-of (x9 / have-org-role-91:ARG1 (x10 / government-organization:wik

Using 7 valid AMR pairs for SMATCH


In [28]:
if len(aligned_gold) == 0:
    print("No valid AMR pairs to score.")
else:
    precision, recall, f1 = score_amr_pairs(
        (penman.encode(g) for g in aligned_gold),
        (penman.encode(p) for p in aligned_pred),
    )
    print("SMATCH Precision:", precision)
    print("SMATCH Recall:", recall)
    print("SMATCH F1:", f1)

Duplicate node name  m  in parsing AMR
Duplicate node name  x0  in parsing AMR


AttributeError: 'NoneType' object has no attribute 'rename_node'